# BirdCam: обучение вида «синица»

Ноутбук для Google Colab. Перед запуском:

1. В BirdCam откройте **Обучение → Подготовить сплит**.
2. На вкладке **Датасет** нажмите **Экспорт ZIP**.
3. Загрузите ZIP в Google Drive как `birdcam-dataset.zip` (или поправьте путь ниже).
4. В Colab включите GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**.

После обучения скачайте `best.onnx`, положите в `%LOCALAPPDATA%\BirdCam\models`,
выберите его в настройках BirdCam и поставьте «Классы птиц» = `0`.

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/dataset
!unzip -q "/content/drive/MyDrive/birdcam-dataset.zip" -d /content/dataset
!ls /content/dataset | head
!cat /content/dataset/data.yaml

In [ ]:
!yolo detect train \
  data=/content/dataset/data.yaml \
  model=yolo26n.pt \
  epochs=80 imgsz=640 batch=16 device=0 \
  project=/content/runs name=birdcam

In [ ]:
!yolo export model=/content/runs/birdcam/weights/best.pt \
  format=onnx opset=18 imgsz=640

from google.colab import files
files.download('/content/runs/birdcam/weights/best.onnx')

## Проверка модели

Можно посмотреть метрики на валидации и предсказания на нескольких кадрах:

```python
from ultralytics import YOLO
model = YOLO('/content/runs/birdcam/weights/best.pt')
model.val(data='/content/dataset/data.yaml')
model.predict(source='/content/dataset/images', conf=0.35, save=True, project='/content/pred')
```

Если `mAP50` высокий, а на новых кадрах модель ошибается — добавьте больше
данных с разных дней и освещения. Одна сцена даёт оптимистичные метрики.